In [5]:
"""
Purpose:
--------
This script compares case-level IDs between two AURORA clinical data files:

1. Metasheet_Analysis.xlsx
   - Contains sample-level IDs in the column: Collaborator_sample_id
   - Example: AUP-ANU1-NB1, AUP-ANU1-NB2

2. AUP_master_annot_sheet_01-09-2026.xlsx
   - Contains patient/case-level IDs in the column: Patient_BCR_Barcode
   - Example: AUP-ANU1

Matching logic:
---------------
The metasheet may contain multiple blood/plasma samples from the same case.
For example:

    AUP-ANU1-NB1
    AUP-ANU1-NB2

Both are counted as one unique case:

    AUP-ANU1

Therefore, the script extracts the case-level ID using the first two parts of
the sample ID pattern: AUP-XXXX.

Main output:
------------
The script prints:

1. Number of unique cases in Metasheet_Analysis.xlsx
2. Number of unique cases in AUP_master_annot_sheet_01-09-2026.xlsx
3. Number of matched unique cases across both files
4. Number of cases present in the metasheet but missing from the master file
5. Number of cases present in the master file but missing from the metasheet
6. Example matched and unmatched case IDs

This is useful for checking whether sample-level sequencing metadata can be
linked correctly to case-level clinical annotation.
"""

import pandas as pd
import re

master_file = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/AUP_master_annot_sheet_01-09-2026.xlsx"
metasheet_file = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/Metasheet_Analysis.xlsx"


# -----------------------------
# Helper function: extract case ID
# Example: AUP-ANU1-NB1 -> AUP-ANU1
# Example: AUP-AQ0Z -> AUP-AQ0Z
# -----------------------------
def extract_case_id(x):
    if pd.isna(x):
        return pd.NA

    x = str(x).strip()

    # Extract AUP-XXXX from IDs such as AUP-ANU1-NB1
    m = re.match(r"^(AUP-[A-Za-z0-9]+)", x)

    return m.group(1) if m else pd.NA


# -----------------------------
# Read Metasheet_Analysis.xlsx
# -----------------------------
meta = pd.read_excel(metasheet_file)

meta["Case_ID"] = meta["Collaborator_sample_id"].apply(extract_case_id)

meta_cases = set(meta["Case_ID"].dropna().unique())


# -----------------------------
# Read AUP master annotation file
# Sheet 1 contains Patient_BCR_Barcode
# -----------------------------
master = pd.read_excel(master_file, sheet_name=0)

# In case duplicate Patient_BCR_Barcode columns exist,
# pandas may rename one as Patient_BCR_Barcode.1
barcode_cols = [c for c in master.columns if str(c).startswith("Patient_BCR_Barcode")]

master_cases = set()

for col in barcode_cols:
    tmp_cases = master[col].apply(extract_case_id).dropna().unique()
    master_cases.update(tmp_cases)


# -----------------------------
# Match unique case IDs
# -----------------------------
matched_cases = meta_cases.intersection(master_cases)

meta_not_in_master = meta_cases - master_cases
master_not_in_meta = master_cases - meta_cases


# -----------------------------
# Print summary
# -----------------------------
print("================ CASE MATCH SUMMARY ================")
print(f"Unique cases in Metasheet_Analysis.xlsx: {len(meta_cases)}")
print(f"Unique cases in AUP_master_annot_sheet_01-09-2026.xlsx: {len(master_cases)}")
print(f"Number of matched unique cases: {len(matched_cases)}")
print(f"Cases in Metasheet but not in AUP master: {len(meta_not_in_master)}")
print(f"Cases in AUP master but not in Metasheet: {len(master_not_in_meta)}")

print("\nFirst 20 Metasheet cases not found in AUP master:")
print(sorted(meta_not_in_master)[:20])


print("\nFirst 20 AUP master cases not found in Metasheet:")
print(sorted(master_not_in_meta)[:20])


# -----------------------------
# Optional check:
# Show how multiple sample IDs map to one case ID
# -----------------------------
print("\n================ SAMPLE-TO-CASE CHECK ================")

print("\nNumber of samples per case in Metasheet:")
print(
    meta.groupby("Case_ID")["Collaborator_sample_id"]
    .nunique()
    .sort_values(ascending=False)
    .head(20)
)

================ CASE MATCH SUMMARY ================
Unique cases in Metasheet_Analysis.xlsx: 270
Unique cases in AUP_master_annot_sheet_01-09-2026.xlsx: 171
Number of matched unique cases: 164
Cases in Metasheet but not in AUP master: 106
Cases in AUP master but not in Metasheet: 7

First 20 Metasheet cases not found in AUP master:
['AUP-AL1Y', 'AUP-AL69', 'AUP-AL9C', 'AUP-AL9W', 'AUP-ALA8', 'AUP-ALBN', 'AUP-AM2M', 'AUP-AMEA', 'AUP-AMOD', 'AUP-AMOL', 'AUP-AMT0', 'AUP-AMUA', 'AUP-AMVG', 'AUP-AMVP', 'AUP-AMW2', 'AUP-AMWC', 'AUP-AMZM', 'AUP-ANEY', 'AUP-ANFX', 'AUP-ANGM']

First 20 AUP master cases not found in Metasheet:
['AUP-AO9O', 'AUP-AON0', 'AUP-AOOA', 'AUP-AOOL', 'AUP-AOY5', 'AUP-APIT', 'AUP-AQ0Z']

================ SAMPLE-TO-CASE CHECK ================

Number of samples per case in Metasheet:
Case_ID
AUP-ANMI    9
AUP-AOFM    9
AUP-ALA9    8
AUP-AON9    8
AUP-APZS    8
AUP-AOGX    8
AUP-AO2R    8
AUP-ANU1    8
AUP-ANYY    8
AUP-ANTU    8
AUP-AO1D    8
AUP-AL9C    8
AUP-AM9R    7


In [7]:
# =====================================================
# Receptor status availability among matched cases
# =====================================================
# Goal:
# -----
# Among cases that matched between Metasheet_Analysis.xlsx
# and AUP_master_annot_sheet_01-09-2026.xlsx, count how many
# unique cases have ER_Status, PR_Status, and HER2_Status available
# in the AUP master annotation file.
#
# Important:
# ----------
# The AUP master file can have multiple rows per case.
# Therefore, we summarize receptor status at the unique Case_ID level.

print("\n================ RECEPTOR STATUS IN MATCHED CASES ================")

# Create row-level Case_ID in the master sheet using Patient_BCR_Barcode column(s)
barcode_cols = [c for c in master.columns if str(c).startswith("Patient_BCR_Barcode")]

master_case_id_df = master[barcode_cols].apply(lambda col: col.apply(extract_case_id))
master["Case_ID"] = master_case_id_df.bfill(axis=1).iloc[:, 0]

# Keep only matched cases
master_matched = master[master["Case_ID"].isin(matched_cases)].copy()

# Clean receptor status values
def clean_status(x):
    if pd.isna(x):
        return pd.NA
    x = str(x).strip()
    if x == "" or x.lower() in ["nan", "na", "n/a", "none"]:
        return pd.NA
    return x

status_cols = ["ER_Status", "PR_Status", "HER2_Status"]

for col in status_cols:
    master_matched[col] = master_matched[col].apply(clean_status)

# Summarize one value per case
# If a case has multiple rows, take the first non-missing value
def first_non_missing(series):
    vals = series.dropna().unique()
    return vals[0] if len(vals) > 0 else pd.NA

case_status = (
    master_matched
    .groupby("Case_ID", as_index=False)[status_cols]
    .agg(first_non_missing)
)

# Count availability
n_matched_cases = len(matched_cases)

n_er = case_status["ER_Status"].notna().sum()
n_pr = case_status["PR_Status"].notna().sum()
n_her2 = case_status["HER2_Status"].notna().sum()

n_all_3 = (
    case_status["ER_Status"].notna()
    & case_status["PR_Status"].notna()
    & case_status["HER2_Status"].notna()
).sum()

print(f"Total matched unique cases: {n_matched_cases}")
print(f"Matched cases with ER_Status available: {n_er}")
print(f"Matched cases with PR_Status available: {n_pr}")
print(f"Matched cases with HER2_Status available: {n_her2}")
print(f"Matched cases with all 3 receptor statuses available: {n_all_3}")



================ RECEPTOR STATUS IN MATCHED CASES ================
Total matched unique cases: 164
Matched cases with ER_Status available: 164
Matched cases with PR_Status available: 164
Matched cases with HER2_Status available: 164
Matched cases with all 3 receptor statuses available: 164


In [10]:
# =====================================================
# Check whether matched cases are present in sheet 2
# =====================================================

print("\n================ MATCHED CASES PRESENT IN SHEET 2 ================")

# Read sheet 2 from AUP master annotation file
master_sheet2 = pd.read_excel(master_file, sheet_name=1)

# Find Patient_BCR_Barcode column(s) in sheet 2
barcode_cols_sheet2 = [
    c for c in master_sheet2.columns
    if str(c).startswith("Patient_BCR_Barcode")
]

# Extract Case_ID from Patient_BCR_Barcode column(s)
master_sheet2_case_id_df = master_sheet2[barcode_cols_sheet2].apply(
    lambda col: col.apply(extract_case_id)
)

master_sheet2["Case_ID"] = master_sheet2_case_id_df.bfill(axis=1).iloc[:, 0]

# Unique cases present in sheet 2
sheet2_cases = set(master_sheet2["Case_ID"].dropna().unique())

# Compare matched cases with sheet 2 cases
matched_cases_in_sheet2 = matched_cases.intersection(sheet2_cases)
matched_cases_not_in_sheet2 = matched_cases - sheet2_cases

print(f"Total matched unique cases: {len(matched_cases)}")
print(f"Matched cases present in sheet 2: {len(matched_cases_in_sheet2)}")
print(f"Matched cases NOT present in sheet 2: {len(matched_cases_not_in_sheet2)}")

print("\nMatched cases NOT present in sheet 2:")
print(sorted(matched_cases_not_in_sheet2))


================ MATCHED CASES PRESENT IN SHEET 2 ================
Total matched unique cases: 164
Matched cases present in sheet 2: 162
Matched cases NOT present in sheet 2: 2

Matched cases NOT present in sheet 2:
['AUP-AMUZ', 'AUP-APD3']


In [11]:
# =====================================================
# Histological Type availability among matched cases
# =====================================================

print("\n================ HISTOLOGICAL TYPE IN MATCHED CASES ================")

# Histological Type is in sheet 2 of AUP_master_annot_sheet_01-09-2026.xlsx
master_hist = pd.read_excel(master_file, sheet_name=1)

# Create Case_ID from Patient_BCR_Barcode column(s)
barcode_cols_hist = [c for c in master_hist.columns if str(c).startswith("Patient_BCR_Barcode")]

master_hist_case_id_df = master_hist[barcode_cols_hist].apply(lambda col: col.apply(extract_case_id))
master_hist["Case_ID"] = master_hist_case_id_df.bfill(axis=1).iloc[:, 0]

# Keep only matched cases
master_hist_matched = master_hist[master_hist["Case_ID"].isin(matched_cases)].copy()

# Clean Histological Type
def clean_histology(x):
    if pd.isna(x):
        return pd.NA
    x = str(x).strip()
    if x == "" or x.lower() in ["nan", "na", "n/a", "none"]:
        return pd.NA
    return x

master_hist_matched["Histological Type"] = master_hist_matched["Histological Type"].apply(clean_histology)

# One histology value per unique case
def first_non_missing(series):
    vals = series.dropna().unique()
    return vals[0] if len(vals) > 0 else pd.NA

case_histology = (
    master_hist_matched
    .groupby("Case_ID", as_index=False)["Histological Type"]
    .agg(first_non_missing)
)

# Count availability
n_matched_cases = len(matched_cases)
n_hist_available = case_histology["Histological Type"].notna().sum()
n_hist_missing = case_histology["Histological Type"].isna().sum()

print(f"Total matched unique cases: {n_matched_cases}")
print(f"Matched cases with Histological Type available: {n_hist_available}")
print(f"Matched cases with Histological Type missing: {n_hist_missing}")

print("\nUnique Histological Type values among matched cases:")
print(case_histology["Histological Type"].dropna().nunique())

print("\nHistological Type counts among matched cases:")
print(
    case_histology["Histological Type"]
    .fillna("Missing/blank")
    .value_counts()
)


================ HISTOLOGICAL TYPE IN MATCHED CASES ================
Total matched unique cases: 164
Matched cases with Histological Type available: 161
Matched cases with Histological Type missing: 1

Unique Histological Type values among matched cases:
4

Histological Type counts among matched cases:
Ductal                      131
Lobular                      16
Mixed Ductal and Lobular      9
Other                         5
Missing/blank                 1
Name: Histological Type, dtype: int64
